In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path
import mdtraj as md


# =============================================================================
# PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(".").resolve()

INPUT_ROOT = (
    PROJECT_ROOT
    / "input"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "output"
)

TOPOLOGY = (
    INPUT_ROOT
    / "md"
    / "topology"
    / "test.gro"
)

TRAJ_277 = (
    INPUT_ROOT
    / "md"
    / "equilibrium-xtc"
    / "trp_whole_dir19.xtc"
)

TRAJ_230 = (
    INPUT_ROOT
    / "md"
    / "equilibrium-xtc"
    / "trp_whole_dir0.xtc"
)

STATE_MODEL_ROOT = (
    INPUT_ROOT
    / "state-model"
    / "t250_k50"
)

PAIRS_DIR = (
    INPUT_ROOT
    / "cooling"
    / "pairs"
)

P_SAMPLES_FILE = (
    STATE_MODEL_ROOT
    / "bmsm"
    / "lag400"
    / "P-samples-lag-400.npz"
)

SASA_DIR = (
    OUTPUT_ROOT
    / "sasa"
)

SASA_FIGURE = (
    SASA_DIR
    / "sasa_scatter.pdf"
)

SASA_FIGURE_SVG = (
    SASA_DIR
    / "sasa_scatter.svg"
)

SASA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# SETTINGS
# =============================================================================

COOLING_RATES = [
    "6e9",
    "6e8",
    "6e7",
    "6e6",
]

N_LAST_FRAMES = 100_000


# =============================================================================
# INPUT VALIDATION
# =============================================================================

required_files = [
    TOPOLOGY,
    TRAJ_277,
    TRAJ_230,
    P_SAMPLES_FILE,
    SASA_DIR / "sasa_residue_info.npz",
]

for short in COOLING_RATES:

    required_files.append(
        SASA_DIR
        / f"end-{short}-sasa-all.npy"
    )

    required_files.append(
        PAIRS_DIR
        / f"{short}_pairs_t250_k50.npy"
    )

missing = [
    path
    for path in required_files
    if not path.is_file()
]

if missing:

    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(
            f"  - {path}"
            for path in missing
        )
    )


# =============================================================================
# PLOTTING STYLE
# =============================================================================

def format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=8,
    tick_labelsize=8,
    title_fontsize=10,
    stroke_in=0.01,
):

    line_width_pts = (
        stroke_in * 72
    )

    ax.minorticks_on()

    if ax.get_xscale() == "log":

        ax.xaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )

    elif minor_ticks_x > 0:

        ax.xaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_x
            )
        )

    if ax.get_yscale() == "log":

        ax.yaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )

    elif minor_ticks_y > 0:

        ax.yaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_y
            )
        )

    for spine in ax.spines.values():

        spine.set_linewidth(
            1.5
        )

        spine.set_color(
            "black"
        )

    ax.tick_params(
        which="major",
        width=line_width_pts,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=line_width_pts * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(
        label_fontsize
    )

    ax.yaxis.label.set_size(
        label_fontsize
    )

    for tick_label in (
        ax.get_xticklabels()
        + ax.get_yticklabels()
    ):

        tick_label.set_fontsize(
            tick_labelsize
        )

    ax.title.set_fontsize(
        title_fontsize
    )


DATA_MARGIN_LEFT_IN = 0.60
DATA_MARGIN_RIGHT_IN = 0.25
DATA_MARGIN_BOTTOM_IN = 0.55
DATA_MARGIN_TOP_IN = 0.30


def make_fixed_axes_figure(
    data_width,
    data_height,
):

    fig_width = (
        DATA_MARGIN_LEFT_IN
        + data_width
        + DATA_MARGIN_RIGHT_IN
    )

    fig_height = (
        DATA_MARGIN_BOTTOM_IN
        + data_height
        + DATA_MARGIN_TOP_IN
    )

    fig = plt.figure(
        figsize=(
            fig_width,
            fig_height,
        )
    )

    ax_left = (
        DATA_MARGIN_LEFT_IN
        / fig_width
    )

    ax_bottom = (
        DATA_MARGIN_BOTTOM_IN
        / fig_height
    )

    ax_width = (
        data_width
        / fig_width
    )

    ax_height = (
        data_height
        / fig_height
    )

    ax = fig.add_axes(
        [
            ax_left,
            ax_bottom,
            ax_width,
            ax_height,
        ]
    )

    return (
        fig,
        ax,
    )


# =============================================================================
# COLOURS
# =============================================================================

vit_colors = {
    "6e9": "#fde725",
    "6e8": "#35b779",
    "6e7": "#31688e",
    "6e6": "#440154",
    "277K": "tab:red",
    "230K": "tab:blue",
}


# =============================================================================
# LOAD RESIDUE INFORMATION
# =============================================================================

residue_info = np.load(
    SASA_DIR
    / "sasa_residue_info.npz",
    allow_pickle=False,
)

resseqs = np.asarray(
    residue_info[
        "resseqs_all"
    ],
    dtype=int,
)

resnames = np.asarray(
    residue_info[
        "residue_names_all"
    ]
).astype(str)

n_residues = len(
    resseqs
)

print(
    f"Residues: {n_residues}"
)


# =============================================================================
# LOAD TPM POSTERIOR SAMPLES
# =============================================================================

with np.load(
    P_SAMPLES_FILE,
    allow_pickle=False,
) as data:

    candidate_keys = []

    for key in data.files:

        arr = np.asarray(
            data[key]
        )

        if (
            arr.ndim == 3
            and arr.shape[1]
            == arr.shape[2]
        ):

            candidate_keys.append(
                key
            )

    if len(candidate_keys) != 1:

        raise ValueError(
            "Expected exactly one "
            "(n_samples, k, k) TPM array in:\n"
            f"{P_SAMPLES_FILE}\n"
            f"Candidates found: {candidate_keys}"
        )

    P_samples = np.asarray(
        data[
            candidate_keys[0]
        ],
        dtype=float,
    ).copy()

print(
    f"TPM samples: "
    f"{P_samples.shape}"
)


# =============================================================================
# STATIONARY DISTRIBUTIONS FOR EACH POSTERIOR TPM
# =============================================================================

def stationary_probability(
    P,
):

    P = np.asarray(
        P,
        dtype=float,
    )

    k = P.shape[0]

    A = (
        P.T
        - np.eye(
            k,
            dtype=float,
        )
    )

    b = np.zeros(
        k,
        dtype=float,
    )

    A[-1, :] = 1.0
    b[-1] = 1.0

    pi, *_ = np.linalg.lstsq(
        A,
        b,
        rcond=None,
    )

    if np.any(
        ~np.isfinite(pi)
    ):

        raise ValueError(
            "Non-finite stationary "
            "distribution encountered."
        )

    minimum = float(
        np.min(pi)
    )

    if minimum < -1e-10:

        raise ValueError(
            "Stationary distribution "
            "contains a substantial "
            "negative probability: "
            f"{minimum:.3e}"
        )

    pi = np.maximum(
        pi,
        0.0,
    )

    pi /= pi.sum()

    return pi


n_samples = (
    P_samples.shape[0]
)

n_states = (
    P_samples.shape[1]
)

stationary_samples = np.empty(
    (
        n_samples,
        n_states,
    ),
    dtype=float,
)

for sample_index in range(
    n_samples
):

    stationary_samples[
        sample_index
    ] = stationary_probability(
        P_samples[
            sample_index
        ]
    )


# Posterior MEAN population.
#
# This replaces the old CSV column:
#
#     lab == "pi_277K"
#     column == "mean"
#
pi_277 = np.mean(
    stationary_samples,
    axis=0,
)

pi_277 /= pi_277.sum()

print(
    f"pi_277 states: {len(pi_277)}"
)

print(
    f"sum(pi_277): {pi_277.sum():.12f}"
)


# =============================================================================
# LOAD COOLED-END SASA AND REWEIGHT USING pi_277
# =============================================================================

def make_snapshot_weights(
    microstate_ids,
    pi,
):

    microstate_ids = np.asarray(
        microstate_ids,
        dtype=int,
    )

    pi = np.asarray(
        pi,
        dtype=float,
    )

    if microstate_ids.size == 0:

        raise ValueError(
            "No endpoint microstates supplied."
        )

    if (
        microstate_ids.min() < 0
        or microstate_ids.max()
        >= len(pi)
    ):

        raise ValueError(
            "Endpoint state ID outside "
            "the range represented by pi_277."
        )

    counts = np.bincount(
        microstate_ids,
        minlength=len(pi),
    )

    factors = np.zeros(
        len(pi),
        dtype=float,
    )

    present = (
        counts > 0
    )

    factors[present] = (
        pi[present]
        / counts[present]
    )

    weights = factors[
        microstate_ids
    ]

    total = weights.sum()

    if total <= 0.0:

        raise ValueError(
            "Snapshot weights have "
            "non-positive total."
        )

    weights /= total

    return weights


sasa_cooled = {}

for short in COOLING_RATES:

    sasa_path = (
        SASA_DIR
        / f"end-{short}-sasa-all.npy"
    )

    pairs_path = (
        PAIRS_DIR
        / f"{short}_pairs_t250_k50.npy"
    )

    sasa_all = np.load(
        sasa_path
    )

    pairs = np.load(
        pairs_path
    )

    if (
        pairs.ndim != 2
        or pairs.shape[1] < 2
    ):

        raise ValueError(
            f"Unexpected pairs shape "
            f"for {short}: {pairs.shape}"
        )

    endpoint_states = (
        pairs[:, 1]
        .astype(int)
    )

    if (
        sasa_all.shape[0]
        != endpoint_states.shape[0]
    ):

        raise ValueError(
            f"{short} length mismatch:\n"
            f"  SASA snapshots: "
            f"{sasa_all.shape[0]}\n"
            f"  endpoint states: "
            f"{endpoint_states.shape[0]}"
        )

    if (
        sasa_all.ndim != 2
        or sasa_all.shape[1]
        != n_residues
    ):

        raise ValueError(
            f"{short} SASA shape mismatch:\n"
            f"  expected second dimension "
            f"{n_residues}\n"
            f"  observed {sasa_all.shape}"
        )

    weights = (
        make_snapshot_weights(
            endpoint_states,
            pi_277,
        )
    )

    sasa_cooled[
        short
    ] = np.average(
        sasa_all,
        axis=0,
        weights=weights,
    )

    print(
        f"{short}: "
        f"{sasa_all.shape[0]} "
        "endpoint snapshots"
    )


# =============================================================================
# EQUILIBRIUM SASA
# =============================================================================

def equilibrium_sasa_last_frames(
    trajectory_path,
    topology_path,
    n_last_frames,
):

    print(
        f"\nLoading equilibrium trajectory:\n"
        f"  {trajectory_path}"
    )

    traj = md.load(
        str(trajectory_path),
        top=str(topology_path),
    )

    n_frames = (
        traj.n_frames
    )

    start = max(
        0,
        n_frames
        - n_last_frames,
    )

    traj = traj[
        start:
    ]

    print(
        f"  total frames: {n_frames}"
    )

    print(
        f"  SASA frames: "
        f"{start}:{n_frames}"
    )

    sasa = md.shrake_rupley(
        traj,
        mode="residue",
    )

    sasa_mean = np.mean(
        sasa,
        axis=0,
    )

    trajectory_resseqs = np.array(
        [
            residue.resSeq
            for residue
            in traj.topology.residues
        ],
        dtype=int,
    )

    if not np.array_equal(
        trajectory_resseqs,
        resseqs,
    ):

        raise ValueError(
            "Residue ordering differs between "
            "equilibrium trajectory topology "
            "and SASA preprocessing."
        )

    return sasa_mean


sasa_277 = (
    equilibrium_sasa_last_frames(
        TRAJ_277,
        TOPOLOGY,
        N_LAST_FRAMES,
    )
)

sasa_230 = (
    equilibrium_sasa_last_frames(
        TRAJ_230,
        TOPOLOGY,
        N_LAST_FRAMES,
    )
)


# =============================================================================
# SASA SCATTER PLOT
# =============================================================================

x = np.arange(
    n_residues
)

xtick_labels = [
    f"{int(resseq)} {name}"
    for resseq, name
    in zip(
        resseqs,
        resnames,
    )
]

offsets = {
    "277K": -0.20,
    "6e9": -0.10,
    "6e8": -0.05,
    "6e7": +0.05,
    "6e6": +0.10,
    "230K": +0.20,
}

fig, ax = (
    make_fixed_axes_figure(
        data_width=6.0,
        data_height=2.4,
    )
)


# 277 K equilibrium
ax.scatter(
    x + offsets["277K"],
    sasa_277,
    label="277 K eq",
    s=12,
    marker="o",
    edgecolors="none",
    alpha=0.9,
    color=vit_colors["277K"],
)


# Cooled ensembles
for short in COOLING_RATES:

    ax.scatter(
        x + offsets[short],
        sasa_cooled[short],
        label=short,
        s=10,
        marker="^",
        edgecolors="none",
        alpha=0.9,
        color=vit_colors[short],
    )


# 230 K equilibrium
ax.scatter(
    x + offsets["230K"],
    sasa_230,
    label="230 K eq",
    s=12,
    marker="s",
    edgecolors="none",
    alpha=0.9,
    color=vit_colors["230K"],
)


# =============================================================================
# AXES
# =============================================================================

ax.set_xlim(
    x.min() - 0.5,
    x.max() + 0.5,
)

ax.set_xlabel(
    "Residue"
)

ax.set_ylabel(
    "SASA (nm$^2$)"
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    xtick_labels,
    rotation=90,
    ha="center",
)

format_axes(
    ax,
    minor_ticks_x=0,
    minor_ticks_y=2,
    label_fontsize=8,
    tick_labelsize=6,
    title_fontsize=10,
    stroke_in=0.01,
)

ax.legend(
    fontsize=6,
    frameon=False,
    loc="best",
)


# =============================================================================
# SAVE
# =============================================================================

fig.savefig(
    SASA_FIGURE
)

fig.savefig(
    SASA_FIGURE_SVG
)

print(
    "\nSaved:"
)

print(
    f"  {SASA_FIGURE}"
)

print(
    f"  {SASA_FIGURE_SVG}"
)

plt.show()